# Football-CV: Nigeria Premier Football League (NPFL) Computer Vision Pipeline
### End-to-End Kaggle GPU Runner for Data Acquisition, Calibration, and YOLOv8 Fine-Tuning

This notebook executes the complete Phase 0 & Phase 1 pipeline on a Kaggle GPU instance:
1. **Environment Setup** (`ultralytics`, `huggingface_hub`, `opencv`, `yt-dlp`)
2. **Data Acquisition**: 1-Click Open Soccer Dataset (HF), SoccerNet-GSR, or NPFL match clips
3. **Unified 4-Class Remapping** (`player`, `goalkeeper`, `referee`, `ball`)
4. **Strict Match-Level Partitioning** (Guaranteed zero frame leakage across train/val/test)
5. **Class Balance Audit & Ball Imbalance Check** (<10% threshold warning)
6. **Pitch Homography Calibration** (Camera pixels $\to$ 105m x 68m pitch coordinates)
7. **YOLOv8 Fine-Tuning** (High-res `imgsz=960` for small ball detection)
8. **Per-Class Metrics (Ball AP50) & ONNX Export**

## 1. Environment Setup & GPU Verification

In [ ]:
# Install dependencies
!pip install -q ultralytics roboflow huggingface_hub pyyaml tqdm pandas opencv-python-headless yt-dlp

import os
import torch
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device Name    : {torch.cuda.get_device_name(0)}")

## 2. API Keys & Authentication (Optional)
Not required for Option A (Open Dataset) or Option B (SoccerNet-GSR). Only required if downloading custom Roboflow Universe projects.

In [ ]:
# Optional: Set Roboflow API key if pulling from Roboflow Universe
os.environ["ROBOFLOW_API_KEY"] = ""

## 3. Phase 0 — Data Acquisition
Choose your data source:
- **Option A (Recommended)**: 1-Click Open Soccer Dataset from HuggingFace (Zero API keys, fast & reliable).
- **Option B**: SoccerNet-GSR GameState subset (`valid.zip` with 5 matches).
- **Option C**: Instant offline smoke-test data generator (60 frames in 5 seconds).
- **Option D**: NPFL match extraction via YouTube and `yt-dlp`.

In [ ]:
# === OPTION A (Recommended): 1-Click Open Soccer YOLO Dataset ===
# Downloads up to 1,000 pre-annotated soccer frames from HuggingFace (No API keys needed)
!python scripts/download_open_dataset.py --max-images 1000

# === OPTION B: SoccerNet-GSR GameState Dataset ===
# Downloads valid.zip and selectively extracts the first 5 matches
# !python scripts/download_soccernet.py --archive valid.zip --max-matches 5

# === OPTION C: Instant Offline Smoke Test ===
# Generates 60 realistic soccer frames with players, goalkeepers, referees & balls
# !python scripts/create_sample_data.py --num-matches 4 --frames-per-match 15

# === OPTION D: Download & Extract NPFL Match Video ===
# Example: Download 3 minutes of an Ikorodu City or Sporting Lagos match from YouTube
# !yt-dlp -f "bestvideo[height<=1080]+bestaudio/best[height<=1080]" --download-sections "*00:15:00-00:18:00" "https://www.youtube.com/watch?v=VIDEO_ID" -o "data/raw/npfl_footage/clip.mp4"
# !python scripts/extract_frames.py --video-path data/raw/npfl_footage/clip.mp4 --venue mobolaji_johnson_arena --cam cam_main --match-id npfl_sample_01

## 4. Phase 0 — 4-Class Remapping & Manifest Generation
Consolidates raw data and translates all labels into the standard schema: `0: player`, `1: goalkeeper`, `2: referee`, `3: ball`. Generates `data/manifest.csv`.

In [ ]:
!python scripts/merge_datasets.py

## 5. Phase 0 — Strict Match-Level Splitting
Partitions the dataset into Train (70%), Val (15%), and Test (15%) splits strictly at the **match level** to guarantee zero video frame leakage.

In [ ]:
!python scripts/split_by_match.py --train-ratio 0.70 --val-ratio 0.15 --test-ratio 0.15

## 6. Phase 0 — Dataset Audit & Small-Object (Ball) Imbalance Check
Audits instance counts for all 4 classes. Issues a critical warning if ball instances are below 10% of player instances.

In [ ]:
!python scripts/validate_dataset.py --data-dir data/labeled

## 7. Phase 0 — Pitch Homography Calibration
Computes a 3x3 projective transformation matrix mapping camera pixels $(u, v)$ to real-world pitch meters $(X, Y)$ on a standard $105 \times 68\text{m}$ pitch.

In [ ]:
# Generates template landmark mappings and computes RANSAC homography matrix
!python scripts/calibrate_pitch.py --venue mobolaji_johnson_arena --cam cam_main

# Run with landmark point file
import os
calib_file = "data/calibration/template_landmarks.json"
if os.path.exists(calib_file):
    !python scripts/calibrate_pitch.py --venue mobolaji_johnson_arena --cam cam_main --points-file data/calibration/template_landmarks.json

## 8. Phase 1 — Detection Model Training (YOLOv8)
Fine-tune YOLOv8 at high resolution (`imgsz=960`) to preserve small ball pixel gradients.

In [ ]:
from ultralytics import YOLO

# Load pretrained YOLOv8s
model = YOLO("yolov8s.pt")

# Train model on 4-class football dataset
results = model.train(
    data="configs/dataset.yaml",
    epochs=50,
    imgsz=960,
    batch=8,
    optimizer="AdamW",
    lr0=0.001,
    mosaic=0.8,
    fliplr=0.5,
    flipud=0.0,
    name="npfl_football_cv",
    project="runs/train"
)

## 9. Phase 1 — Per-Class Metrics Inspection
Monitor class-specific AP50. Specifically watch **Class 3 (Ball)** to ensure small-object convergence.

In [ ]:
# Evaluate best checkpoint
metrics = model.val()

class_names = ["player", "goalkeeper", "referee", "ball"]
print("\n================ PER-CLASS AP50 METRICS ================")
if hasattr(metrics.box, "ap50"):
    for idx, name in enumerate(class_names):
        if idx < len(metrics.box.ap50):
            print(f"  [{idx}] {name:<12}: AP50 = {metrics.box.ap50[idx]:.4f}")
print(f"  Aggregate mAP50    : {metrics.box.map50:.4f}")
print(f"  Aggregate mAP50-95 : {metrics.box.map:.4f}")
print("=========================================================")

## 10. Phase 1 — Export to ONNX for High-Speed Inference

In [ ]:
# Export fine-tuned weights to ONNX format
onnx_path = model.export(format="onnx", imgsz=960, dynamic=True)
print(f"[+] Exported ONNX model to: {onnx_path}")